# Tennis Predictor — Data Exploration

Initial exploration notebook to verify data loaded correctly and get a feel for the dataset.

**Prerequisites:**
- Database populated via `tennis-predictor load-data`
- Environment configured (`.env` file present)

In [ ]:
from tennis_predictor.data.storage import get_session
from sqlalchemy import text
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Basic counts

In [ ]:
with get_session() as session:
    summary = pd.read_sql(
        text("""
            SELECT tour,
                   COUNT(*) AS total_matches,
                   COUNT(DISTINCT winner_id) AS unique_winners,
                   COUNT(DISTINCT tournament_id) AS unique_tournaments,
                   MIN(match_date) AS earliest,
                   MAX(match_date) AS latest
            FROM matches
            GROUP BY tour
        """),
        session.bind,
    )

summary

## 2. Matches per year per surface (sanity check)

In [ ]:
with get_session() as session:
    by_year = pd.read_sql(
        text("""
            SELECT EXTRACT(YEAR FROM match_date)::int AS year,
                   tour,
                   surface,
                   COUNT(*) AS matches
            FROM matches
            GROUP BY year, tour, surface
            ORDER BY year, tour, surface
        """),
        session.bind,
    )

by_year.head(20)

## 3. Top players by match count

In [ ]:
with get_session() as session:
    top_players = pd.read_sql(
        text("""
            SELECT p.name_full, p.tour, p.country_code,
                   COUNT(m.match_id) AS matches,
                   SUM(CASE WHEN m.winner_id = p.player_id THEN 1 ELSE 0 END) AS wins
            FROM players p
            JOIN matches m ON p.player_id IN (m.winner_id, m.loser_id)
            GROUP BY p.player_id, p.name_full, p.tour, p.country_code
            ORDER BY matches DESC
            LIMIT 20
        """),
        session.bind,
    )

top_players['win_pct'] = (top_players['wins'] / top_players['matches'] * 100).round(1)
top_players

## 4. Data quality checks

In [ ]:
with get_session() as session:
    quality = pd.read_sql(
        text("""
            SELECT
                COUNT(*) AS total,
                SUM(CASE WHEN surface IS NULL THEN 1 ELSE 0 END) AS missing_surface,
                SUM(CASE WHEN minutes IS NULL THEN 1 ELSE 0 END) AS missing_minutes,
                SUM(CASE WHEN winner_rank IS NULL THEN 1 ELSE 0 END) AS missing_winner_rank,
                SUM(CASE WHEN retirement THEN 1 ELSE 0 END) AS retirements,
                SUM(CASE WHEN walkover THEN 1 ELSE 0 END) AS walkovers
            FROM matches
        """),
        session.bind,
    )

quality.T

## 5. Match stats coverage by year

Sackmann data has stats from ~1991+ for ATP. Older matches may lack detail.

In [ ]:
with get_session() as session:
    stats_coverage = pd.read_sql(
        text("""
            SELECT EXTRACT(YEAR FROM m.match_date)::int AS year,
                   m.tour,
                   COUNT(DISTINCT m.match_id) AS total_matches,
                   COUNT(DISTINCT ms.match_id) AS matches_with_stats,
                   ROUND(
                     COUNT(DISTINCT ms.match_id)::numeric / NULLIF(COUNT(DISTINCT m.match_id), 0) * 100,
                     1
                   ) AS coverage_pct
            FROM matches m
            LEFT JOIN match_stats ms ON m.match_id = ms.match_id
            GROUP BY year, m.tour
            ORDER BY year DESC, m.tour
            LIMIT 30
        """),
        session.bind,
    )

stats_coverage